# Data Cleaning – Olist E-Commerce
Simple step by step cleaning: missing values, wrong values, duplicates, data types and text, outliers. The clean tables are saved to `data/processed/`.

In [46]:
import kagglehub
import pandas as pd

# Download latest version
path = kagglehub.dataset_download("olistbr/brazilian-ecommerce")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'brazilian-ecommerce' dataset.
Path to dataset files: /kaggle/input/brazilian-ecommerce


In [64]:
import numpy as np

customers      = pd.read_csv(f"{path}/olist_customers_dataset.csv", dtype={"customer_zip_code_prefix": str})
sellers        = pd.read_csv(f"{path}/olist_sellers_dataset.csv", dtype={"seller_zip_code_prefix": str})
orders         = pd.read_csv(f"{path}/olist_orders_dataset.csv")
order_items    = pd.read_csv(f"{path}/olist_order_items_dataset.csv")
order_payments = pd.read_csv(f"{path}/olist_order_payments_dataset.csv")
order_reviews  = pd.read_csv(f"{path}/olist_order_reviews_dataset.csv")
products       = pd.read_csv(f"{path}/olist_products_dataset.csv")
category_translation = pd.read_csv(f"{path}/product_category_name_translation.csv", encoding="utf-8-sig")

In [65]:
print("customers      ", len(customers))
print("sellers        ", len(sellers))
print("orders         ", len(orders))
print("order_items    ", len(order_items))
print("order_payments ", len(order_payments))
print("order_reviews  ", len(order_reviews))
print("products       ", len(products))
print("category_translation", len(category_translation))

customers       99441
sellers         3095
orders          99441
order_items     112650
order_payments  103886
order_reviews   99224
products        32951
category_translation 71


# 1. Missing values

In [66]:
print("missing before:", products["product_category_name"].isnull().sum())
products["product_category_name"] = products["product_category_name"].fillna("unknown")
print("missing after :", products["product_category_name"].isnull().sum())

missing before: 610
missing after : 0


In [67]:
cols = ["product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm"]

for col in cols:
    products.loc[products[col] <= 0, col] = np.nan                       # 0 -> missing
    median_by_category = products.groupby("product_category_name")[col].transform("median")
    products[col] = products[col].fillna(median_by_category)             # median of the category
    products[col] = products[col].fillna(products[col].median())         # if the whole category is empty

print(products[cols].isnull().sum())

product_weight_g     0
product_length_cm    0
product_height_cm    0
product_width_cm     0
dtype: int64


In [68]:
order_reviews["review_comment_title"] = order_reviews["review_comment_title"].replace(r"^\s*$", np.nan, regex=True)
order_reviews["review_comment_message"] = order_reviews["review_comment_message"].replace(r"^\s*$", np.nan, regex=True)

# 2. Wrong values

In [69]:
bad = (order_payments["payment_type"] == "not_defined") | (order_payments["payment_value"] <= 0) | order_payments["payment_value"].isnull()
print("rows removed:", bad.sum())
order_payments = order_payments[~bad]

rows removed: 9


In [70]:
print("rows fixed:", (order_payments["payment_installments"] < 1).sum())
order_payments.loc[order_payments["payment_installments"] < 1, "payment_installments"] = 1

rows fixed: 2


In [71]:
bad = order_items["price"].isnull() | (order_items["price"] <= 0) | (order_items["freight_value"] < 0)
print("rows removed:", bad.sum())
order_items = order_items[~bad]

rows removed: 0


In [72]:
bad = order_reviews["review_score"].isnull() | (order_reviews["review_score"] < 1) | (order_reviews["review_score"] > 5)
print("rows removed:", bad.sum())
order_reviews = order_reviews[~bad]

rows removed: 0


# 3. Duplicates

In [73]:
print("customers      ", customers.duplicated().sum())
print("sellers        ", sellers.duplicated().sum())
print("orders         ", orders.duplicated().sum())
print("order_items    ", order_items.duplicated().sum())
print("order_payments ", order_payments.duplicated().sum())
print("order_reviews  ", order_reviews.duplicated().sum())
print("products       ", products.duplicated().sum())
print("category_translation", category_translation.duplicated().sum())

customers      = customers.drop_duplicates()
sellers        = sellers.drop_duplicates()
orders         = orders.drop_duplicates()
order_items    = order_items.drop_duplicates()
order_payments = order_payments.drop_duplicates()
order_reviews  = order_reviews.drop_duplicates()
products       = products.drop_duplicates()
category_translation = category_translation.drop_duplicates()

customers       0
sellers         0
orders          0
order_items     0
order_payments  0
order_reviews   0
products        0
category_translation 0


In [74]:
customers      = customers.drop_duplicates(subset="customer_id")
sellers        = sellers.drop_duplicates(subset="seller_id")
orders         = orders.drop_duplicates(subset="order_id")
products       = products.drop_duplicates(subset="product_id")
category_translation = category_translation.drop_duplicates(subset="product_category_name")
order_items    = order_items.drop_duplicates(subset=["order_id", "order_item_id"])
order_payments = order_payments.drop_duplicates(subset=["order_id", "payment_sequential"])

In [75]:
print("extra reviews:", order_reviews["order_id"].duplicated().sum())

order_reviews = order_reviews.sort_values(["order_id", "review_answer_timestamp"])
order_reviews = order_reviews.drop_duplicates(subset="order_id", keep="last")

print("one review per order:", order_reviews["order_id"].is_unique)

extra reviews: 551
one review per order: True


# 4. Data types and text

In [76]:
date_cols = ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
             "order_delivered_customer_date", "order_estimated_delivery_date"]

for col in date_cols:
    orders[col] = pd.to_datetime(orders[col], errors="coerce")

order_items["shipping_limit_date"] = pd.to_datetime(order_items["shipping_limit_date"], errors="coerce")
order_reviews["review_creation_date"] = pd.to_datetime(order_reviews["review_creation_date"], errors="coerce")
order_reviews["review_answer_timestamp"] = pd.to_datetime(order_reviews["review_answer_timestamp"], errors="coerce")

In [77]:
approved_before_purchase  = orders["order_approved_at"] < orders["order_purchase_timestamp"]
delivered_before_purchase = orders["order_delivered_customer_date"] < orders["order_purchase_timestamp"]
carrier_before_purchase   = orders["order_delivered_carrier_date"] < orders["order_purchase_timestamp"]
carrier_after_delivery    = orders["order_delivered_carrier_date"] > orders["order_delivered_customer_date"]

print("approved before purchase :", approved_before_purchase.sum())
print("delivered before purchase:", delivered_before_purchase.sum())
print("carrier before purchase  :", carrier_before_purchase.sum())
print("carrier after delivery   :", carrier_after_delivery.sum())

orders.loc[approved_before_purchase, "order_approved_at"] = pd.NaT
orders.loc[delivered_before_purchase, "order_delivered_customer_date"] = pd.NaT
orders.loc[carrier_before_purchase | carrier_after_delivery, "order_delivered_carrier_date"] = pd.NaT

approved before purchase : 0
delivered before purchase: 0
carrier before purchase  : 166
carrier after delivery   : 23


In [78]:
customers["customer_city"]  = customers["customer_city"].str.strip().str.lower()
customers["customer_state"] = customers["customer_state"].str.strip().str.upper()
sellers["seller_city"]      = sellers["seller_city"].str.strip().str.lower()
sellers["seller_state"]     = sellers["seller_state"].str.strip().str.upper()
orders["order_status"]      = orders["order_status"].str.strip().str.lower()
order_payments["payment_type"] = order_payments["payment_type"].str.strip().str.lower()

customers["customer_zip_code_prefix"] = customers["customer_zip_code_prefix"].str.strip().str.zfill(5)
sellers["seller_zip_code_prefix"]     = sellers["seller_zip_code_prefix"].str.strip().str.zfill(5)

In [79]:
products = products.rename(columns={"product_name_lenght": "product_name_length",
                                    "product_description_lenght": "product_description_length"})

In [80]:
new_rows = pd.DataFrame({
    "product_category_name": ["pc_gamer", "portateis_cozinha_e_preparadores_de_alimentos", "unknown"],
    "product_category_name_english": ["pc_gamer", "portable_kitchen_and_food_preparers", "unknown"],
})
category_translation = pd.concat([category_translation, new_rows], ignore_index=True)
category_translation = category_translation.drop_duplicates(subset="product_category_name")

In [81]:
orders = orders[orders["customer_id"].isin(customers["customer_id"])]

order_items = order_items[order_items["order_id"].isin(orders["order_id"])]
order_items = order_items[order_items["product_id"].isin(products["product_id"])]
order_items = order_items[order_items["seller_id"].isin(sellers["seller_id"])]

order_payments = order_payments[order_payments["order_id"].isin(orders["order_id"])]
order_reviews  = order_reviews[order_reviews["order_id"].isin(orders["order_id"])]

# 5. Outliers

In [82]:
for name, values in [("price", order_items["price"]),
                     ("freight_value", order_items["freight_value"]),
                     ("payment_value", order_payments["payment_value"])]:
    Q1 = values.quantile(0.25)
    Q3 = values.quantile(0.75)
    IQR = Q3 - Q1
    upper = Q3 + 1.5 * IQR
    count = (values > upper).sum()
    print(f"{name:15s} upper limit = {upper:8.2f}   outliers = {count}   max = {values.max():.2f}")

price           upper limit =   277.40   outliers = 8427   max = 6735.00
freight_value   upper limit =    33.25   outliers = 11613   max = 409.68
payment_value   upper limit =   344.37   outliers = 7981   max = 13664.08


In [83]:
print("orders -> customers  :", orders["customer_id"].isin(customers["customer_id"]).all())
print("items -> orders      :", order_items["order_id"].isin(orders["order_id"]).all())
print("items -> products    :", order_items["product_id"].isin(products["product_id"]).all())
print("items -> sellers     :", order_items["seller_id"].isin(sellers["seller_id"]).all())
print("payments -> orders   :", order_payments["order_id"].isin(orders["order_id"]).all())
print("reviews -> orders    :", order_reviews["order_id"].isin(orders["order_id"]).all())
print("products -> category :", products["product_category_name"].isin(category_translation["product_category_name"]).all())
print("one review per order :", order_reviews["order_id"].is_unique)

orders -> customers  : True
items -> orders      : True
items -> products    : True
items -> sellers     : True
payments -> orders   : True
reviews -> orders    : True
products -> category : True
one review per order : True


In [84]:
print("customers      ", len(customers))
print("sellers        ", len(sellers))
print("orders         ", len(orders))
print("order_items    ", len(order_items))
print("order_payments ", len(order_payments))
print("order_reviews  ", len(order_reviews))
print("products       ", len(products))
print("category_translation", len(category_translation))

customers       99441
sellers         3095
orders          99441
order_items     112650
order_payments  103877
order_reviews   98673
products        32951
category_translation 74


In [85]:
import os
os.makedirs("data/processed", exist_ok=True)

for col in date_cols:
    orders[col] = orders[col].dt.strftime("%Y-%m-%d %H:%M:%S")
order_items["shipping_limit_date"] = order_items["shipping_limit_date"].dt.strftime("%Y-%m-%d %H:%M:%S")
order_reviews["review_creation_date"] = order_reviews["review_creation_date"].dt.strftime("%Y-%m-%d %H:%M:%S")
order_reviews["review_answer_timestamp"] = order_reviews["review_answer_timestamp"].dt.strftime("%Y-%m-%d %H:%M:%S")

customers.to_csv("data/processed/customers.csv", index=False)
sellers.to_csv("data/processed/sellers.csv", index=False)
orders.to_csv("data/processed/orders.csv", index=False)
order_items.to_csv("data/processed/order_items.csv", index=False)
order_payments.to_csv("data/processed/order_payments.csv", index=False)
order_reviews.to_csv("data/processed/order_reviews.csv", index=False)
products.to_csv("data/processed/products.csv", index=False)
category_translation.to_csv("data/processed/category_translation.csv", index=False)

print("saved in data/processed/")

saved in data/processed/


In [ ]:
# Colab only: download the clean files as a zip (does nothing on your own computer)
try:
    import shutil
    from google.colab import files
    shutil.make_archive("processed_data", "zip", "data/processed")
    files.download("processed_data.zip")
except ImportError:
    pass